<a href="https://colab.research.google.com/github/Achmad96/anomaly-fraud-detection/blob/master/fraud_detection_on_transactions_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade tensorflow==2.19.0

In [ ]:
!pip install -U kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 2.6 MB/s eta 0:00:00


In [ ]:
!pip install torch torch-geometric scikit-learn matplotlib seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.4 MB/s eta 0:00:00


In [1]:
import tensorflow as tf
print(tf.__version__)

2.19.0


# Variables Initialization

In [1]:
from google.colab import drive
from pathlib import Path

drive_path = Path('/content/drive')
drive.mount(str(drive_path))
drive_path /= 'MyDrive'

Mounted at /content/drive


In [2]:
SEED = 42
THRESHOLD = 0.5
DATASET_ROOT = drive_path / 'datasets' /'fraud_dataset'
MODEL_PATH = DATASET_ROOT / 'fraud_detection_model.keras'
SCALER_PATH = DATASET_ROOT / 'fraud_scaler.pkl'
MODEL_CHECKPOINT = DATASET_ROOT / 'fraud_detection_model_checkpoint.keras'

# Load Dataset

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils import shuffle

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import plot_model
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers, Model, models, callbacks, regularizers
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, LSTM, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from scipy.spatial.distance import pdist, squareform
import networkx as nx

from imblearn.over_sampling import SMOTE

In [5]:
df = pd.read_csv(DATASET_ROOT/'synthetic_transactions.csv')
df.head()

,timestamp,transaction_id,customer_id,transaction_total,merchant_category,payment_method,region,total_quantity,total_quantity_7d,reported_transaction_total,is_fraud,reasoning
0,2024-09-14,661dded2,76f6e63b3e664d3daefdf63e4f511646,6643972,Clothing,Credit Card,Surabaya,46,50.00,6643972,0,Neither condition was met. 0 equals 0 (no disc...
1,2024-12-30,08a19d6a,70153d9ed06641c688cf3f18d15d0b78,5348844,Groceries,Debit Card,Bandung,43,23.76,5348844,0,The first condition (0 != 0) is impossible as ...
2,2025-01-28,0e7b5c27,83f7d3ae3e6c4539a48acd59b1d4ae5e,1523903,Groceries,Bank Transfer,Surabaya,4,4.17,1523903,0,Neither condition is met. The first condition ...
3,2025-09-22,aba521c0,0ecfff90aae346999281c584944b241e,8429969,Clothing,Credit Card,Bandung,40,38.46,8429969,0,Neither condition was met. The value 0 is equa...
4,2021-06-08,7a022f30,b5bc28295cee4ed7abe7efee8dce89f9,2370554,Clothing,Bank Transfer,Bandung,36,33.96,2370554,0,Neither condition is met. The first condition ...


In [6]:
legit_count = df[df['is_fraud'] == 0]['transaction_id'].nunique()
fraud_count = df[df['is_fraud'] == 1]['transaction_id'].nunique()
transaction_count = df['transaction_id'].nunique()
fraud_percentages = (fraud_count / transaction_count) * 100
print(f"Legitimate Transactions: {legit_count}")
print(f"Fraudulent Transactions: {fraud_count}")
print(f'Fraud percentages: {fraud_percentages:.2f}%')

Legitimate Transactions: 896
Fraudulent Transactions: 104
Fraud percentages: 10.40%


# Exploratory Data Analysis

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   timestamp                   1000 non-null   object 
 1   transaction_id              1000 non-null   object 
 2   customer_id                 1000 non-null   object 
 3   transaction_total           1000 non-null   int64  
 4   merchant_category           1000 non-null   object 
 5   payment_method              1000 non-null   object 
 6   region                      1000 non-null   object 
 7   total_quantity              1000 non-null   int64  
 8   total_quantity_7d           1000 non-null   float64
 9   reported_transaction_total  1000 non-null   int64  
 10  is_fraud                    1000 non-null   int64  
 11  reasoning                   1000 non-null   object 
dtypes: float64(1), int64(4), object(7)
memory usage: 93.9+ KB


In [8]:
df.isnull().sum()

,0
timestamp,0
transaction_id,0
customer_id,0
transaction_total,0
merchant_category,0
payment_method,0
region,0
total_quantity,0
total_quantity_7d,0
reported_transaction_total,0


In [9]:
df.describe().round(3)

,transaction_total,total_quantity,total_quantity_7d,reported_transaction_total,is_fraud
count,1000.000,1000.000,1000.000,1000.000,1000.000
mean,5103297.698,50.639,44.599,4939854.970,0.104
std,2898513.507,28.557,35.107,2907284.041,0.305
min,4391.000,1.000,0.610,4391.000,0.000
25%,2548357.500,26.000,17.830,2386015.000,0.000
50%,5340687.000,51.000,36.875,5085380.500,0.000
75%,7539517.000,76.000,60.392,7352765.500,0.000
max,9987150.000,100.000,188.680,9987150.000,1.000


In [10]:
payment_frauds = df.drop_duplicates(subset=['transaction_id'])[df['is_fraud'] == 1]['payment_method'].value_counts()
payment_frauds.index
fig = px.bar(
    payment_frauds,
    title='Fraud by Customer Payment Methods',
    x=payment_frauds.index,
    y=payment_frauds,
    labels={'x': 'Payment Methods', 'y': 'Counts'},
    hover_name=payment_frauds.index.astype(str),
    width=500, height=400,
    color=payment_frauds.index.astype(str),
    custom_data=['count'],
    color_discrete_sequence=px.colors.qualitative.T10)
fig.update_traces(hovertemplate="<br>".join(["Count: %{customdata[0]}"]))
fig.show()

In [11]:
merchant_category_frauds = df.drop_duplicates(subset=['transaction_id'])[df['is_fraud'] == 1]['merchant_category'].value_counts()
fig = px.bar(
    merchant_category_frauds,
    title='Fraud By Merchant Categories',
    x=merchant_category_frauds.index,
    y=merchant_category_frauds.to_numpy(),
    labels={'x': 'Merchant Categories', 'y': 'Counts'},
    hover_name=merchant_category_frauds.index,
    width=500, height=400,
    color=merchant_category_frauds.index,
    custom_data=['count'],
    color_discrete_sequence=px.colors.qualitative.T10)
fig.update_traces(hovertemplate="<br>".join(["Count: %{customdata[0]}"]))
fig.show()

In [12]:
quantity_spike_ratio = df['total_quantity']/df['total_quantity_7d']
df[quantity_spike_ratio > 2]['is_fraud'].value_counts()

,count
is_fraud,
1,69


In [13]:
transaction_total_delta = df['reported_transaction_total'] - df['transaction_total']
df[transaction_total_delta < 0]['is_fraud'].value_counts()

,count
is_fraud,
1,56


In [15]:
df[(transaction_total_delta < 0) & (quantity_spike_ratio > 2)]['is_fraud'].value_counts()

,count
is_fraud,
1,25


# Feature Engineering

In [35]:
df.columns

Index(['timestamp', 'transaction_id', 'customer_id', 'transaction_total',
       'merchant_category', 'payment_method', 'region', 'total_quantity',
       'total_quantity_7d', 'reported_transaction_total', 'is_fraud',
       'reasoning'],
      dtype='object')

In [36]:
numerical_features = df.select_dtypes(include=['int64', 'float64'])
categorical_features = df.select_dtypes(include=['object'])
print(f"""
Numerical Features:
{numerical_features.columns}

Categorical Features:
{categorical_features.columns}
""")


Numerical Features:
Index(['transaction_total', 'total_quantity', 'total_quantity_7d',
       'reported_transaction_total', 'is_fraud'],
      dtype='object')

Categorical Features:
Index(['transaction_id', 'customer_id', 'merchant_category', 'payment_method',
       'region', 'reasoning'],
      dtype='object')



In [37]:
X = df.drop(columns=['is_fraud'], inplace=False)
y = df['is_fraud']

In [38]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
X['merchant_category'] = encoder.fit_transform(df['merchant_category'])
X['payment_method'] = encoder.fit_transform(df['payment_method'])
X['region'] = encoder.fit_transform(df['region'])

In [39]:
X.head()

,timestamp,transaction_id,customer_id,transaction_total,merchant_category,payment_method,region,total_quantity,total_quantity_7d,reported_transaction_total,reasoning
0,2024-09-14,661dded2,76f6e63b3e664d3daefdf63e4f511646,6643972,0,1,4,46,50.00,6643972,Neither condition was met. 0 equals 0 (no disc...
1,2024-12-30,08a19d6a,70153d9ed06641c688cf3f18d15d0b78,5348844,2,2,0,43,23.76,5348844,The first condition (0 != 0) is impossible as ...
2,2025-01-28,0e7b5c27,83f7d3ae3e6c4539a48acd59b1d4ae5e,1523903,2,0,4,4,4.17,1523903,Neither condition is met. The first condition ...
3,2025-09-22,aba521c0,0ecfff90aae346999281c584944b241e,8429969,0,1,0,40,38.46,8429969,Neither condition was met. The value 0 is equa...
4,2021-06-08,7a022f30,b5bc28295cee4ed7abe7efee8dce89f9,2370554,0,0,0,36,33.96,2370554,Neither condition is met. The first condition ...


In [40]:
X['quantity_spike_ratio'] = quantity_spike_ratio

In [41]:
X['transaction_total_delta'] = transaction_total_delta

In [42]:
# X['hour'] = pd.DataFrame(df['timestamp'].dtype)
df['timestamp'] = pd.to_datetime(df['timestamp'])
X['hour'] = df['timestamp'].dt.hour

In [43]:
X['hour'].unique()

array([0], dtype=int32)

In [44]:
feature_cols=['quantity_spike_ratio',	'transaction_total_delta']
# feature_cols=['quantity_spike_ratio',	'transaction_total_delta']

In [45]:
X = X[feature_cols]

In [46]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   quantity_spike_ratio     1000 non-null   float64
 1   transaction_total_delta  1000 non-null   int64  
dtypes: float64(1), int64(1)
memory usage: 15.8 KB


In [47]:
y.info()

<class 'pandas.core.series.Series'>
RangeIndex: 1000 entries, 0 to 999
Series name: is_fraud
Non-Null Count  Dtype
--------------  -----
1000 non-null   int64
dtypes: int64(1)
memory usage: 7.9 KB


In [48]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Class distribution - Train: {np.bincount(y_train)}")
print(f"Class distribution - Test: {np.bincount(y_test)}")

Training set: (700, 2)
Test set: (300, 2)
Class distribution - Train: [627  73]
Class distribution - Test: [269  31]


In [49]:
smote = SMOTE(random_state=SEED, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
X_train_resampled, y_train_resampled = shuffle(X_train_resampled, y_train_resampled, random_state=SEED)
print(f"X-train-original: {X_train.shape}, X-train-resampled: {X_train_resampled.shape}")
print(f"y-train-original: {y_train.shape}, y-train-resampled: {y_train_resampled.shape}")

X-train-original: (700, 2), X-train-resampled: (1254, 2)
y-train-original: (700,), y-train-resampled: (1254,)


In [50]:
fraud_indicators = y_train_resampled.value_counts()
fraud_indicators.index = ['Non-Fraud', 'Fraud']
fig = px.bar(
    fraud_indicators,
    x=fraud_indicators.index,
    y=fraud_indicators.to_numpy(),
    labels={'x': 'Fraud Indicator', 'y': 'Count'},
    hover_name=fraud_indicators.index,
    color=fraud_indicators.index,
    width=500, height=400,
    hover_data=['count'],
    title='Class Distribution after Oversampling',
    color_discrete_sequence=px.colors.qualitative.T10)
fig.update_traces(hovertemplate="<br>".join(["Count: %{customdata[0]}"]))
fig.show()

In [51]:
X_train_resampled.head()

,quantity_spike_ratio,transaction_total_delta
1253,4.096401,0
101,0.529983,0
51,1.499864,0
63,0.520009,0
1069,2.485067,0


In [52]:
# scaler = RobustScaler()
# X_train_scaled = scaler.fit_transform(X_train_resampled)
# X_test_scaled = scaler.transform(X_test)
X_train_scaled = X_train_resampled
X_test_scaled = X_test

In [53]:
num_nodes = X_train_scaled.shape[0]
input_dim = X_train_scaled.shape[1]
input_dim

2

# Modelling

In [54]:
model = models.Sequential([
    # Conv Block 1
    layers.Conv1D(32, kernel_size=3, activation='relu', padding='same', input_shape=(input_dim, 1)),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(0.3),

    # Conv Block 2
    layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=1), # Because the feature is too small, pool_size is adjusted to 1
    layers.Dropout(0.3),

    # Conv Block 3
    layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling1D(),

    # Dense layers
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    # Output
    layers.Dense(1, activation='sigmoid')
])

In [55]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

In [56]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 2, 32)          │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 2, 32)          │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 1, 64)          │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1, 64)          │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 1, 128)         │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 97,985 (382.75 KB)

 Trainable params: 97,537 (381.00 KB)

 Non-trainable params: 448 (1.75 KB)

In [57]:
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
model_checkpoint = ModelCheckpoint(
    filepath=MODEL_CHECKPOINT,
    save_weights_only=False,
    monitor='val_loss', # Monitor the validation loss
    mode='min', # Mode 'min' means the monitored quantity should be minimized (for loss)
    save_best_only=True,
    # verbose=1 # Optional: logs a message when a new best model is saved
)


In [58]:
model.fit(X_train_scaled, y_train_resampled,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    validation_data=(X_test_scaled, y_test),
    callbacks=[early_stop, reduce_lr, model_checkpoint]
  )

Epoch 1/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - accuracy: 0.5574 - auc: 0.4899 - loss: 0.8251 - val_accuracy: 0.8967 - val_auc: 0.8177 - val_loss: 0.6513 - learning_rate: 1.0000e-05
Epoch 2/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5789 - auc: 0.5836 - loss: 0.7392 - val_accuracy: 0.9067 - val_auc: 0.9689 - val_loss: 0.6044 - learning_rate: 1.0000e-05
Epoch 3/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6061 - auc: 0.5978 - loss: 0.7258 - val_accuracy: 0.9200 - val_auc: 0.9868 - val_loss: 0.5615 - learning_rate: 1.0000e-05
Epoch 4/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.6435 - auc: 0.6376 - loss: 0.6801 - val_accuracy: 0.9200 - val_auc: 0.9847 - val_loss: 0.5183 - learning_rate: 1.0000e-05
Epoch 5/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.6667 - auc: 0.6942 - loss: 0.6442 - val_accuracy: 0.9200 - val_auc: 0.9938 - val_loss: 0.4762 - learning_rate: 1.0000e-05
Epoch 6/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.

In [59]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

history = model.history.history
epochs = list(range(len(history['loss'])))

fig = make_subplots(rows=1, cols=3, subplot_titles=('Loss', 'Accuracy','Auc'))

# Loss Plot
# Trace for 'loss'
fig.add_trace(
    go.Scatter(
        x=epochs,
        y=history['loss'],
        mode='lines',
        name='Training Loss',
        line=dict(color='blue')
    ),
    row=1, col=1
)
# Trace for 'val_loss' with fill
fig.add_trace(
    go.Scatter(
        x=epochs,
        y=history['val_loss'],
        mode='lines',
        name='Validation Loss',
        fill='tonexty',
        fillcolor='rgba(0,0,255,0.1)',
        line=dict(color='red')
    ),
    row=1, col=1
)

# Accuracy Plot
# Trace for 'accuracy'
fig.add_trace(
    go.Scatter(
        x=epochs,
        y=history['accuracy'],
        mode='lines',
        name='Training Accuracy',
        line=dict(color='green')
    ),
    row=1, col=2
)
# Trace for 'val_accuracy' with fill
fig.add_trace(
    go.Scatter(
        x=epochs,
        y=history['val_accuracy'],
        mode='lines',
        name='Validation Accuracy',
        fill='tonexty',
        fillcolor='rgba(0,255,0,0.1)',
        line=dict(color='orange')
    ),
    row=1, col=2
)

# Area Under the Curve (AUC) Plot
# Trace for 'auc'
fig.add_trace(
    go.Scatter(
        x=epochs,
        y=history['auc'],
        mode='lines',
        name='Training AUC',
        line=dict(color='green')
    ),
    row=1, col=3
)
# Trace for 'val_auc' with fill
fig.add_trace(
    go.Scatter(
        x=epochs,
        y=history['val_auc'],
        mode='lines',
        name='Validation AUC',
        fill='tonexty',
        fillcolor='rgba(128,0,128,0.1)',
        line=dict(color='purple')
    ),
    row=1, col=3
)

fig.update_layout(height=400, showlegend=True)
fig.show()

In [60]:
y_pred =  model.predict(X_test).flatten()
y_pred = (y_pred > THRESHOLD).astype('int32')

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step


In [61]:
y_test.shape

(300,)

In [62]:
cm = confusion_matrix(y_test, y_pred)
cm

array([[269,   0],
       [  8,  23]])

In [63]:
print("CNN Classification Report:")
print(classification_report(y_test.to_numpy().astype('float32'), y_pred))

CNN Classification Report:
              precision    recall  f1-score   support

         0.0       0.97      1.00      0.99       269
         1.0       1.00      0.74      0.85        31

    accuracy                           0.97       300
   macro avg       0.99      0.87      0.92       300
weighted avg       0.97      0.97      0.97       300



In [64]:
cm = confusion_matrix(y_test, y_pred)
fig = px.imshow(
    cm,
    text_auto=True,
    labels=dict(x="Predicted", y="True", color="Count"),
    x=['Non-Fraud', 'Fraud'],
    y=['Non-Fraud', 'Fraud'],
    color_continuous_scale='Blues',
    title='Confusion Matrix',
    width=500, height=400
)
fig.show()

# Saving Model

In [50]:
model.save(MODEL_PATH)
print(f'Model saved at {MODEL_PATH}')

Model saved at /content/drive/MyDrive/datasets/fraud_dataset/fraud_detection_model.keras


# Feature Importants

In [74]:
# import shap
# import numpy as np

# background = X_train_resampled[:100]
# explainer = shap.DeepExplainer(model, background)
# raw_shap_values = explainer.shap_values(X_test[:100])

# if isinstance(raw_shap_values, list):
#     squeezed_shap_values = np.squeeze(raw_shap_values[0])
# else:
#     squeezed_shap_values = np.squeeze(raw_shap_values)

# squeezed_features = np.squeeze(X_test[:100])

# shap.summary_plot(squeezed_shap_values, squeezed_features, feature_names=feature_cols)

# Inferences

In [126]:
inference_df = {
    'merchant_category': [
        1, 2, 4, 3, 1, 2, 1, 5, 2, 1,
        3, 4, 5, 2, 1, 4, 3, 2, 1, 1,
        2, 1, 5, 3, 1, 2, 4, 1, 1, 3,
        2, 1, 4, 1, 2, 1, 3, 5, 2, 1,
        4, 5, 1, 2, 1, 3, 2, 4, 5, 2
    ],
    'payment_method': [
        1, 1, 3, 2, 1, 1, 1, 4, 2, 1,
        1, 2, 4, 1, 1, 3, 1, 2, 1, 1,
        1, 1, 4, 2, 1, 1, 3, 1, 2, 1,
        1, 1, 4, 2, 1, 1, 1, 4, 2, 1,
        1, 4, 1, 2, 1, 1, 1, 2, 3, 1
    ],
    'region': [
        1, 1, 4, 2, 1, 3, 1, 5, 2, 1,
        1, 3, 5, 1, 2, 4, 1, 3, 1, 2,
        1, 1, 5, 3, 2, 1, 5, 1, 2, 3,
        1, 1, 4, 2, 1, 3, 1, 5, 2, 1,
        1, 5, 2, 1, 3, 1, 1, 2, 5, 3
    ],
    'quantity_spike_ratio': [
        1.0, 1.1, 3.5, 0.9, 1.0, 1.2, 1.0, 4.2, 1.1, 1.0,
        1.0, 1.0, 2.8, 1.1, 1.0, 3.1, 1.0, 1.2, 1.0, 1.0,
        1.1, 1.0, 4.5, 1.0, 1.2, 1.0, 3.8, 1.0, 1.1, 1.0,
        1.0, 1.0, 2.9, 1.0, 1.1, 1.0, 1.0, 4.0, 1.2, 1.0,
        1.0, 4.1, 1.0, 1.1, 1.0, 1.0, 1.0, 1.2, 3.6, 1.0
    ],
    'transaction_total_delta': [
        0, 0, 45, 2, 0, 5, 0, 75, 0, 0,
        0, 0, 38, 0, 0, 50, 4, 0, 0, 0,
        0, 0, 80, 0, 6, 0, 62, 0, 0, 0,
        0, 0, 42, 0, 0, 0, 0, 68, 5, 0,
        0, 70, 0, 0, 0, 0, 0, 8, 55, 0
    ],
    'is_fraud': [
        0, 0, 1, 0, 0, 0, 0, 1, 0, 0,
        0, 0, 1, 0, 0, 1, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 1, 0, 0,
        0, 1, 0, 0, 0, 0, 0, 0, 1, 0
    ]
}
inference_df = pd.DataFrame(inference_df)
y_inference = inference_df['is_fraud']
X_inference = inference_df[feature_cols]
X_inference_scaled = X_inference
y_pred_prob = model.predict(X_inference_scaled)
y_inf_pred = (y_pred_prob > THRESHOLD).astype('int32')
y_pred_prob = y_pred_prob.flatten()
y_pred_prob

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step


array([0.02646406, 0.02728465, 1.        , 0.6614299 , 0.02646406,
       0.9999472 , 0.02646406, 1.        , 0.02728465, 0.02646406,
       0.02646406, 0.02646406, 1.        , 0.02728465, 0.02646406,
       1.        , 0.9989227 , 0.02848927, 0.02646406, 0.02646406,
       0.02728465, 0.02646406, 1.        , 0.02646406, 0.99999714,
       0.02646406, 1.        , 0.02646406, 0.02728465, 0.02646406,
       0.02646406, 0.02646406, 1.        , 0.02646406, 0.02728465,
       0.02646406, 0.02646406, 1.        , 0.9999472 , 0.02646406,
       0.02646406, 1.        , 0.02646406, 0.02728465, 0.02646406,
       0.02646406, 0.02646406, 1.        , 1.        , 0.02646406],
      dtype=float32)

In [127]:
confusion_matrix(y_inference, y_inf_pred)

array([[34,  6],
       [ 0, 10]])

In [128]:
accuracy_score(y_inference, y_inf_pred)

0.88

In [131]:
roc_auc_score(y_inference, y_pred_prob)

np.float64(0.9874999999999999)

In [132]:
print(classification_report(y_inference.to_numpy().astype('float32'), y_inf_pred))

              precision    recall  f1-score   support

         0.0       1.00      0.85      0.92        40
         1.0       0.62      1.00      0.77        10

    accuracy                           0.88        50
   macro avg       0.81      0.93      0.84        50
weighted avg       0.93      0.88      0.89        50

